# Clase 199 — FastAPI sirviendo modelos

Construye un servicio FastAPI completo (predict, predict-batch, health, metrics) y lo loadtestea con un cliente sincrónico.

Requiere: `pip install fastapi uvicorn[standard] joblib scikit-learn httpx`. Si querés correr el server en background desde el notebook, levantalo en otra terminal.

## Setup

In [ ]:
import os, shutil, tempfile, joblib
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

WORK = Path(tempfile.gettempdir()) / 'fastapi_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

X, y = load_iris(return_X_y=True)
m = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, y)
joblib.dump(m, 'model.pkl')
print('modelo guardado:', Path('model.pkl').stat().st_size, 'bytes')

## 1. `app.py` — servicio FastAPI

In [ ]:
app_src = '''\
from contextlib import asynccontextmanager
from typing import Annotated
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator
import joblib, numpy as np, time, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("app")

@asynccontextmanager
async def lifespan(app: FastAPI):
    log.info("loading model...")
    app.state.model = joblib.load("model.pkl")
    app.state.start = time.time()
    log.info("model loaded.")
    yield
    log.info("shutting down.")

app = FastAPI(title="iris-api", version="1.0.0", lifespan=lifespan)

class IrisIn(BaseModel):
    features: Annotated[list[float], Field(min_length=4, max_length=4)]
    @field_validator("features")
    @classmethod
    def positive(cls, v):
        if any(x < 0 for x in v): raise ValueError("features must be non-negative")
        return v

class IrisOut(BaseModel):
    cls: int
    proba: list[float]

class BatchIn(BaseModel):
    rows: list[list[float]]

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": app.state.model is not None,
            "uptime_s": round(time.time() - app.state.start, 1)}

@app.post("/predict", response_model=IrisOut)
def predict(x: IrisIn):
    arr = np.asarray(x.features).reshape(1, -1)
    cls = int(app.state.model.predict(arr)[0])
    proba = app.state.model.predict_proba(arr)[0].tolist()
    return IrisOut(cls=cls, proba=proba)

@app.post("/predict-batch")
def predict_batch(b: BatchIn):
    arr = np.asarray(b.rows)
    if arr.shape[1] != 4:
        raise HTTPException(422, "each row needs 4 features")
    cls = app.state.model.predict(arr).tolist()
    return {"n": len(cls), "predictions": cls}
'''
Path('app.py').write_text(app_src)
print('app.py escrito —', len(app_src.splitlines()), 'líneas')

## 2. Server en background + smoke test

Levantamos `uvicorn` como subprocess y le pegamos con `httpx`.

In [ ]:
import subprocess, time, httpx, sys
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '8765', '--log-level', 'warning'],
    cwd=str(WORK), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
# Esperar a que arranque
for _ in range(30):
    try:
        r = httpx.get('http://127.0.0.1:8765/health', timeout=0.5)
        if r.status_code == 200: break
    except Exception: time.sleep(0.2)
print('health:', r.json())

In [ ]:
# Predict OK
print(httpx.post('http://127.0.0.1:8765/predict', json={'features': [5.1, 3.5, 1.4, 0.2]}).json())

# Predict con valor negativo → 422
r = httpx.post('http://127.0.0.1:8765/predict', json={'features': [-1, 2, 3, 4]})
print(r.status_code, r.json())

# Batch
rows = X[:100].tolist()
r = httpx.post('http://127.0.0.1:8765/predict-batch', json={'rows': rows})
print('batch ok, predicciones:', r.json()['n'])

## 3. Loadtest: 1 batch vs 100 requests individuales

In [ ]:
import time
rows = X[:100].tolist()

t0 = time.perf_counter()
for r in rows:
    httpx.post('http://127.0.0.1:8765/predict', json={'features': r})
t_individual = time.perf_counter() - t0

t0 = time.perf_counter()
httpx.post('http://127.0.0.1:8765/predict-batch', json={'rows': rows})
t_batch = time.perf_counter() - t0

print(f'100 requests individuales: {t_individual * 1000:.1f} ms')
print(f'1 request batch (100):    {t_batch * 1000:.1f} ms')
print(f'speedup batch: {t_individual / t_batch:.1f}x')

In [ ]:
# Cleanup
proc.terminate(); proc.wait(timeout=5)
print('server detenido.')

## Ejercicio guiado

1. Agregá `prometheus-fastapi-instrumentator` y un endpoint `GET /metrics`. Verificá con `curl localhost:8765/metrics | grep http_request`.
2. Convertí `/predict` a `async def` y simulá un I/O-bound con `await asyncio.sleep(0.01)` (un "feature store call"). Compará throughput con 50 clientes concurrentes vs el `def` original.
3. Escribí un `locustfile.py` con 100 users y corré 60 s. Reportá p50/p95/p99.
4. Hardenelo para producción: `FastAPI(docs_url=None, redoc_url=None)`, agregá rate limiting con `slowapi`, y log estructurado JSON con `structlog`.

## Conclusiones

- `lifespan` evita re-cargar el modelo por request — diferencia de 100× en latencia.
- Pydantic v2 valida el input gratis (sin schema custom) y bloquea malos requests con 422.
- Batching reduce overhead de HTTP + permite vectorización numpy.
- Sin healthcheck honesto, K8s no sabe cuándo sacar al pod del LB.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. FastAPI/uvicorn no están instalados, así que **el código de la app se muestra completo y correcto** (lo copiás a `app.py` y lo levantás con `uvicorn app:app --reload`), y para que la celda **corra igual** ejecutamos la *lógica* que la app envuelve: la predicción, la carga única del modelo (singleton del `lifespan`), el speedup del batching, la concurrencia `async`, y el healthcheck. Un endpoint FastAPI no es más que esa lógica + validación Pydantic + routing.

In [ ]:
import numpy as np, time, asyncio
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

Xi, yi = load_iris(return_X_y=True)
_iris_model = LogisticRegression(max_iter=500).fit(Xi, yi)
print('modelo iris entrenado (lo que la app cargaría en el lifespan).')

### Ejercicio 1 — API mínima `POST /predict`

Código FastAPI completo (con Pydantic) + la lógica de predicción ejecutable y verificada.

In [ ]:
app_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI()
model = joblib.load("model.pkl")

class IrisInput(BaseModel):
    features: list[float]

class IrisOutput(BaseModel):
    prediction: int
    proba: list[float]

@app.post("/predict", response_model=IrisOutput)
def predict(payload: IrisInput):
    import numpy as np
    x = np.array(payload.features).reshape(1, -1)
    cls = int(model.predict(x)[0])
    proba = model.predict_proba(x)[0].tolist()
    return IrisOutput(prediction=cls, proba=proba)
'''
# concepto ejecutable: la lógica del handler
def predict_logic(features):
    x = np.array(features).reshape(1, -1)
    return int(_iris_model.predict(x)[0]), _iris_model.predict_proba(x)[0].tolist()

cls, proba = predict_logic([5.1, 3.5, 1.4, 0.2])
print('POST /predict {"features":[5.1,3.5,1.4,0.2]} ->', {'prediction': cls, 'proba': [round(p, 3) for p in proba]})
assert cls in (0, 1, 2) and abs(sum(proba) - 1.0) < 1e-6
print('OK — /docs (Swagger) probaría exactamente esta lógica.')

### Ejercicio 2 — `lifespan`: cargar el modelo UNA vez

El modelo se carga en un `lifespan` async y se guarda en `app.state.model`: se ejecuta una vez al arrancar, no por request. Verificamos el "cargó una sola vez" con un contador.

In [ ]:
lifespan_code = '''
from contextlib import asynccontextmanager
from fastapi import FastAPI
import joblib

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.model = joblib.load("model.pkl")   # UNA vez, al startup
    print("model loaded")
    yield
    app.state.model = None                        # cleanup al shutdown

app = FastAPI(lifespan=lifespan)
'''
# concepto: singleton — el loader corre una vez aunque haya N requests
load_counter = {'n': 0}
_cache = {}
def get_model():
    if 'model' not in _cache:
        load_counter['n'] += 1
        _cache['model'] = _iris_model
    return _cache['model']

for _ in range(100):
    get_model()                        # 100 "requests"
assert load_counter['n'] == 1, 'el modelo debe cargarse una sola vez'
print('requests:', 100, '| cargas de modelo:', load_counter['n'], '=> singleton OK.')

### Ejercicio 3 — Batching: 100 predicciones sueltas vs 1 batch de 100

Vectorizar en un solo `predict` amortiza el overhead por llamada. Medimos ambos y confirmamos el speedup.

In [ ]:
batch = Xi[:100]
t0 = time.perf_counter()
for row in batch:
    _iris_model.predict(row.reshape(1, -1))
t_individual = time.perf_counter() - t0

t0 = time.perf_counter()
_iris_model.predict(batch)             # 1 sola llamada vectorizada
t_batch = time.perf_counter() - t0

print(f'100 sueltas : {t_individual*1e3:7.2f} ms')
print(f'1 batch(100): {t_batch*1e3:7.2f} ms')
print(f'speedup     : {t_individual / t_batch:6.1f}x')
assert t_batch < t_individual, 'el batch debería amortizar el overhead por llamada'
print('OK — /predict-batch (BatchInput.rows) sirve más throughput con menos overhead.')

### Ejercicio 4 — `async def` vs `def` con una llamada externa

Para I/O (llamar otra API), `async def` libera el event loop y permite concurrencia; `def` bloquea un worker del thread pool. Simulamos 10 llamadas externas de 50 ms: secuencial (~500 ms) vs concurrente con `asyncio.gather` (~50 ms).

In [ ]:
external_code = '''
import httpx
@app.get("/enrich")
async def enrich():
    async with httpx.AsyncClient() as client:
        r = await client.get("https://api.example.com/features")   # NO bloquea el loop
    return r.json()
'''
async def fake_external_call():
    await asyncio.sleep(0.05)          # 50 ms de "red"
    return 1

async def sequential():
    return [await fake_external_call() for _ in range(10)]

async def concurrent():
    return await asyncio.gather(*[fake_external_call() for _ in range(10)])

t0 = time.perf_counter(); asyncio.run(sequential()); t_seq = time.perf_counter() - t0
t0 = time.perf_counter(); asyncio.run(concurrent()); t_conc = time.perf_counter() - t0
print(f'secuencial (bloqueante): {t_seq*1e3:6.0f} ms')
print(f'async gather           : {t_conc*1e3:6.0f} ms')
assert t_conc < t_seq / 2, 'async debería ser mucho más rápido con I/O concurrente'
print('OK — async def libera el event loop mientras espera I/O.')

### Ejercicio 5 — Observabilidad: `/health` y `/metrics`

`/health` devuelve estado + si el modelo cargó; `prometheus-fastapi-instrumentator` expone `/metrics`. Implementamos el health y un contador de requests estilo Prometheus.

In [ ]:
obs_code = '''
from prometheus_fastapi_instrumentator import Instrumentator
Instrumentator().instrument(app).expose(app)   # crea /metrics

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": app.state.model is not None}
'''
def health():
    return {'status': 'ok', 'model_loaded': _cache.get('model') is not None}

request_counter = {'/predict': 0}
def observe(endpoint):
    request_counter[endpoint] = request_counter.get(endpoint, 0) + 1

for _ in range(5):
    predict_logic([5.1, 3.5, 1.4, 0.2]); observe('/predict')

print('GET /health ->', health())
print('metric http_requests_total{endpoint="/predict"} =', request_counter['/predict'])
assert health()['model_loaded'] is True and request_counter['/predict'] == 5
print('OK — health + métrica de requests listos para scrapeo Prometheus.')